In [2]:
# SPARKSESSION
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Tugas11-KopiNusantara') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
spark.conf.set('spark.sql.shuffle.partitions', '4')

print('SparkSession siap. Versi Spark:', spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [3]:
# Membuat dataset 
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [4]:
# Membaca CSV ke Spark
df = spark.read.csv('tugas11_penjualan.csv', header=True, inferSchema=True)
df.printSchema()
df.show(5)
print('Jumlah baris:', df.count())

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Jumlah baris: 2500


In [5]:
# A. Feature Engineering 
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Model indexer disimpan karena dipakai lagi di bagian E
idx_menu_model = StringIndexer(inputCol='nama_menu', outputCol='nama_menu_idx').fit(df)
idx_cabang_model = StringIndexer(inputCol='cabang', outputCol='cabang_idx').fit(df)
idx_waktu_model = StringIndexer(inputCol='waktu_pesan', outputCol='waktu_pesan_idx').fit(df)

df_idx = idx_menu_model.transform(df)
df_idx = idx_cabang_model.transform(df_idx)
df_idx = idx_waktu_model.transform(df_idx)

kolom_fitur = ['qty', 'harga_satuan', 'nama_menu_idx', 'cabang_idx', 'waktu_pesan_idx']
assembler = VectorAssembler(inputCols=kolom_fitur, outputCol='features')
df_final = assembler.transform(df_idx)

df_final.select('nama_menu', 'nama_menu_idx', 'cabang', 'cabang_idx',
                'waktu_pesan', 'waktu_pesan_idx', 'features', 'total_penjualan').show(5, truncate=False)

+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|nama_menu     |nama_menu_idx|cabang    |cabang_idx|waktu_pesan|waktu_pesan_idx|features                 |total_penjualan   |
+--------------+-------------+----------+----------+-----------+---------------+-------------------------+------------------+
|Kentang Goreng|5.0          |Semarang  |2.0       |Malam      |1.0            |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Sore       |3.0            |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|Kentang Goreng|5.0          |Semarang  |2.0       |Siang      |2.0            |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|Kentang Goreng|5.0          |Yogyakarta|1.0       |Pagi       |0.0            |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|Roti Bakar    |4.0          |Magelang  |0.0       |Malam      |1.0            |[2.0,30000.0,4.0,0.0,1.0]|58157.944120

In [6]:
# B. Train/Test Split dan Pelatihan Model 
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f'Jumlah data training: {train.count()}')
print(f'Jumlah data testing : {test.count()}')

lr = LinearRegression(featuresCol='features', labelCol='total_penjualan')
model = lr.fit(train)

print('\nModel berhasil dilatih!')
for nama, koef in zip(kolom_fitur, model.coefficients):
    print(f'Koefisien {nama:<16}: {koef:>12,.2f}')
print(f'Intercept                   : {model.intercept:>12,.2f}')

Jumlah data training: 1839
Jumlah data testing : 661

Model berhasil dilatih!
Koefisien qty             :    22,046.66
Koefisien harga_satuan    :         3.02
Koefisien nama_menu_idx   :      -123.21
Koefisien cabang_idx      :       279.28
Koefisien waktu_pesan_idx :       339.64
Intercept                   :   -66,903.18


In [7]:
# C. Evaluasi Model 
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import avg

pred = model.transform(test)

rmse = RegressionEvaluator(labelCol='total_penjualan', predictionCol='prediction', metricName='rmse').evaluate(pred)
r2 = RegressionEvaluator(labelCol='total_penjualan', predictionCol='prediction', metricName='r2').evaluate(pred)
rata_total = test.select(avg('total_penjualan')).first()[0]

print(f'RMSE : Rp {rmse:,.0f}')
print(f'R2   : {r2:.4f}  ({r2*100:.1f}% variasi total_penjualan dijelaskan model)')
print(f'Rata-rata total_penjualan data testing: Rp {rata_total:,.0f}')
print(f'RMSE relatif terhadap rata-rata       : {rmse/rata_total*100:.1f}%')

if r2 >= 0.8:
    print('\nInterpretasi: R2 tinggi, model cukup baik untuk memprediksi total_penjualan.')
elif r2 >= 0.5:
    print('\nInterpretasi: R2 sedang, model cukup membantu tetapi masih perlu diperbaiki.')
else:
    print('\nInterpretasi: R2 rendah, model belum layak dipakai.')

RMSE : Rp 10,901
R2   : 0.9107  (91.1% variasi total_penjualan dijelaskan model)
Rata-rata total_penjualan data testing: Rp 65,244
RMSE relatif terhadap rata-rata       : 16.7%

Interpretasi: R2 tinggi, model cukup baik untuk memprediksi total_penjualan.


In [8]:
# D. Eksperimen Perbandingan Fitur 
# Model kedua: hanya qty dan harga_satuan, memakai baris train/test yang sama
assembler_dasar = VectorAssembler(inputCols=['qty', 'harga_satuan'], outputCol='features_dasar')
train_d = assembler_dasar.transform(train)
test_d = assembler_dasar.transform(test)

lr_d = LinearRegression(featuresCol='features_dasar', labelCol='total_penjualan')
model_d = lr_d.fit(train_d)
pred_d = model_d.transform(test_d)

rmse_d = RegressionEvaluator(labelCol='total_penjualan', predictionCol='prediction', metricName='rmse').evaluate(pred_d)
r2_d = RegressionEvaluator(labelCol='total_penjualan', predictionCol='prediction', metricName='r2').evaluate(pred_d)

print('Model'.ljust(28) + 'R2'.rjust(10) + 'RMSE (Rp)'.rjust(16))
print('Utama (5 fitur)'.ljust(28) + f'{r2:>10.4f}' + f'{rmse:>16,.0f}')
print('Kedua (qty + harga_satuan)'.ljust(28) + f'{r2_d:>10.4f}' + f'{rmse_d:>16,.0f}')

selisih = r2 - r2_d
print(f'\nSelisih R2 (utama - kedua): {selisih:+.4f}')
if abs(selisih) < 0.01:
    print('Interpretasi: selisih sangat kecil, menu/cabang/waktu tidak meningkatkan akurasi secara berarti.')
elif selisih > 0:
    print('Interpretasi: model utama lebih baik, menu/cabang/waktu memberi tambahan akurasi.')
else:
    print('Interpretasi: model sederhana justru lebih baik, fitur tambahan tidak membantu.')

Model                               R2       RMSE (Rp)
Utama (5 fitur)                 0.9107          10,901
Kedua (qty + harga_satuan)      0.9108          10,896

Selisih R2 (utama - kedua): -0.0001
Interpretasi: selisih sangat kecil, menu/cabang/waktu tidak meningkatkan akurasi secara berarti.


In [9]:
# E. Prediksi Skenario Baru 
from pyspark.sql.functions import col

skenario = pd.DataFrame({
    'nama_menu':    ['Kopi Susu', 'Kentang Goreng', 'Es Teh'],
    'cabang':       ['Magelang', 'Yogyakarta', 'Semarang'],
    'waktu_pesan':  ['Pagi', 'Siang', 'Malam'],
    'qty':          [2, 5, 1],
    'harga_satuan': [20000, 25000, 15000],
})
df_skenario = spark.createDataFrame(skenario)

# Pakai model StringIndexer yang SAMA dari bagian A, bukan StringIndexer baru
df_skenario = idx_menu_model.transform(df_skenario)
df_skenario = idx_cabang_model.transform(df_skenario)
df_skenario = idx_waktu_model.transform(df_skenario)
df_skenario = assembler.transform(df_skenario)

hasil = model.transform(df_skenario).withColumn('qty_x_harga', col('qty') * col('harga_satuan'))
hasil.select('nama_menu', 'cabang', 'waktu_pesan', 'qty', 'harga_satuan',
             'qty_x_harga', 'prediction').show(truncate=False)

+--------------+----------+-----------+---+------------+-----------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|qty_x_harga|prediction        |
+--------------+----------+-----------+---+------------+-----------+------------------+
|Kopi Susu     |Magelang  |Pagi       |2  |20000       |40000      |37525.35198441948 |
|Kentang Goreng|Yogyakarta|Siang      |5  |25000       |125000     |119245.65122961697|
|Es Teh        |Semarang  |Malam      |1  |15000       |15000      |1015.8637051739497|
+--------------+----------+-----------+---+------------+-----------+------------------+

